# Quantum-Enhanced Classifier

This notebook implements a hybrid quantum-classical neural network to classify network traffic as 'attack' or 'normal'.

The workflow is as follows:
1. Load the 10-dimensional latent features created by the `autoencoder_pca_qml_preprocessing.ipynb` notebook.
2. Load the original dataset to get the corresponding labels.
3. Define a quantum feature map using PennyLane, which encodes the features into the rotation angles of qubits.
4. Build a hybrid model consisting of the quantum circuit and a classical neural network with a softmax output.
5. Train the model on the latent features and labels.
6. Evaluate the classifier's performance.

## 2. Imports

In [8]:
import pennylane as qml
from pennylane import numpy as np
import torch
from torch import nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pathlib import Path

## 3. Load Data

In [9]:
base = Path('.')
latent_features_path = base / 'outputs' / 'autoencoder_pca_representation.csv'
original_data_path = base / 'outputs' / 'classical_qi_predictions_all_splits.csv'

# Load the 10-dimensional features
X = pd.read_csv(latent_features_path).values

# Load the original data to get the labels
df_orig = pd.read_csv(original_data_path)
y_strings = df_orig['scenario'] # Assuming 'scenario' contains 'attack' or 'normal'

# Encode labels to integers (0 and 1)
le = LabelEncoder()
y = le.fit_transform(y_strings)

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Classes: {le.classes_}")

Features shape: (48, 10)
Labels shape: (48,)
Classes: ['attack' 'normal']


## 4. Define the Quantum Circuit

In [ ]:
n_qubits = 10
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface='torch')
def quantum_feature_map(inputs):
    """The quantum feature map circuit."""
    # Encode features as rotation angles
    # Using inputs[..., i] supports parameter broadcasting (both 1D and 2D batch inputs)
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)
        qml.Hadamard(wires=i)
    
    # Cascade of entangling CNOT gates
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])
    
    # Return the expectation value of the PauliZ operator for each qubit
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

## 5. Build the Hybrid Quantum-Classical Model

In [11]:
class HybridModel(nn.Module):
    def __init__(self, n_qubits=10, dropout_rate=0.5):
        super().__init__()
        self.q_layer = qml.qnn.TorchLayer(quantum_feature_map, weight_shapes={})
        
        self.c_layers = nn.Sequential(
            # Layer 1
            nn.Linear(n_qubits, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(dropout_rate),
            
            # Layer 2
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(dropout_rate),
            
            # Output Layer
            nn.Linear(32, 2)
        )
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # Ensure input is contiguous
        x = x.contiguous()
        x = self.q_layer(x)
        x = self.c_layers(x)
        return self.softmax(x)

## 6. Train the Model

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to torch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# Hyperparameters
epochs = 50
learning_rate = 0.1
batch_size = 8

# Model, loss, and optimizer
model = HybridModel()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(epochs):
    for i in range(0, len(X_train_t), batch_size):
        batch_X = X_train_t[i:i+batch_size]
        batch_y = y_train_t[i:i+batch_size]

        # Drop the last batch if it's smaller than the specified batch_size
        if len(batch_X) != batch_size:
            continue

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    # Validation
    with torch.no_grad():
        test_outputs = model(X_test_t)
        _, predicted = torch.max(test_outputs, 1)
        accuracy = (predicted == y_test_t).sum().item() / len(y_test_t)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}, Test Accuracy: {accuracy:.4f}')

IndexError: index 8 is out of bounds for dimension 0 with size 8

## 7. Final Evaluation

In [ ]:
with torch.no_grad():
    outputs = model(X_test_t)
    _, predicted = torch.max(outputs, 1)
    
    print(f"Final Test Accuracy: {(predicted == y_test_t).sum().item() / len(y_test_t):.4f}")
    print("\nClassification Report:")
    # For a more detailed report, you can use sklearn's classification_report
    from sklearn.metrics import classification_report
    print(classification_report(y_test_t.numpy(), predicted.numpy(), target_names=le.classes_))